ssh tmtong@ugcpu1.cse.ust.hk
cd /project/fyp24_ho3/tmtong/llama/data/ImageNet1k
conda activate muxit

In [11]:
import os
os.listdir()
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# t1 = torch.load("image_features/train/train_features_0.pt")
# t2 = torch.load("image_features/train/train_features_1.pt")

# tensor = torch.cat((t1, t2), 0)
# tensor[:1000].save("experimental_image_features/train/train_features_0.pt")

# t1.shape, t2.shape

## since the features are off by 1, we need to fix that
count_old, count_new = 0, 0
tensor = torch.load(f"image_features/validation/validation_features_0.pt").to(device)
count_old += tensor.shape[0]
for i in range(1, 13):
    t2 = torch.load(f"image_features/validation/validation_features_{i}.pt").to(device)
    count_old += t2.shape[0]
    tensor = torch.cat((tensor, t2), 0)
    torch.save(tensor[:1000], f"experimental_image_features/validation/validation_features_{i-1}.pt")
    count_new += tensor[:1000].shape[0]
    tensor = tensor[1000:]
    print(f"old: {count_old}, new: {count_new}")


/tmp/ipykernel_248504/2720952673.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tensor = torch.load(f"image_features/validation/validation_features_0.pt").to(device)
/

old: 1001, new: 1000
old: 2001, new: 2000
old: 3001, new: 3000
old: 4001, new: 4000
old: 5001, new: 5000
old: 6001, new: 6000
old: 7001, new: 7000
old: 8001, new: 8000
old: 9001, new: 9000
old: 10001, new: 10000
old: 11001, new: 11000


FileNotFoundError: [Errno 2] No such file or directory: 'image_features/validation/validation_features_12.pt'

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import argparse
import torch
# from torchvision import transforms
from datasets import load_dataset
from transformers import AutoProcessor, MllamaVisionModel
from tqdm import tqdm
from PIL import Image

parser = argparse.ArgumentParser()
parser.add_argument("split", "-s", "--split", type=str, default=None, nargs="?", help="Dataset split to use: train, validation, test")
parser.add_argument("skip", "-s", "--skip", type=int, default=None, nargs="?", help="Skip first n samples from the dataset")
parser.add_argument("take", "-t", "--take", type=int, default=None, nargs="?", help="First n samples to take from the dataset (after skipping)")
args = parser.parse_args()

def arg_check(args):
    if args.split not in ["train", "validation", "test"] or args.split is not None:
        raise ValueError("Invalid split argument. Must be one of: train, validation, test")
    if args.skip and args.skip < 0 or args.skip is not None:
        raise ValueError("Invalid skip argument. Must be a positive integer")
    if args.take and args.take < 0 or args.take is not None:
        raise ValueError("Invalid take argument. Must be a positive integer")
    return args

args = arg_check(args)
if args.split is None:
    args.split = "train"
dataset = load_dataset("ILSVRC/imagenet-1k", split=args.split, streaming=True)
if args.skip:
    dataset = dataset.skip(args.skip)
if args.take:
    dataset = dataset.take(args.take)

model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
model_path = "../../model/"

processor = AutoProcessor.from_pretrained(model_path)
model = MllamaVisionModel.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# def custom_collate_fn(data):
#     # data = Image.open(data[0]["image"]).convert("RGB") if isinstance(#something here) else #something here
#     return data
# #something here about data transformation

def infer(processor, model, data):
    with torch.no_grad():
        inputs = processor(
            images=data["image"],
            return_tensors="pt",
        ).to(model.device)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        outputs = model(**inputs).last_hidden_state[0]
        outputs = outputs.mean(dim=1).mean(dim=1)
        return outputs
        
with torch.inference_mode():
    tensor = torch.tensor([]).to(model.device)
    for i, data in enumerate(tqdm(dataset)):
        try:
            # inputs = processor(
            #     images=data["image"],
            #     return_tensors="pt",
            # ).to(model.device)
            # inputs = {k: v.to(model.device) for k, v in inputs.items()}
            # outputs = model(**inputs).last_hidden_state[0]
            # outputs = outputs.mean(dim=1).mean(dim=1)
            outputs = infer(processor, model, data)
            tensor = torch.cat((tensor, outputs), 0)
            if not i % 1000:
                torch.save(tensor, f"train_features_cp_{i//1000}.pt")
                print(f"checkpoint {i//1000} saved")
                tensor = torch.tensor([]).to(model.device)
        except Exception as e:
            with open("train_features_error.txt", "a") as f:
                f.write(f"{i}: {e}\n")

Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

0it [00:00, ?it/s]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=817x363 at 0x7F0E00F49A10>, 'label': 726}


0it [00:02, ?it/s]

tensor([[ 0.7148,  0.4082,  0.4648,  ..., -1.1719, -3.0625, -2.2656]],
       device='cuda:0')
0
